In [ ]:

# # 1-a dalis: ECG Denoising Pipeline test with a single .npy file from DATA/ZIVE_DATA
# Uses CONFIG/pipeline_config.yaml

# https://chatgpt.com/c/6925c2c6-e564-8327-bf65-a8fdeeea1cdd

# "Run ECG denoising pipeline, ECG ectopy detecting and removing pipeline,
# HRV calculation on a single .npy ECG file.\n"

from datetime import datetime, timedelta

# from calendar import monthrange
from pathlib import Path
import pandas as pd
import math, time
from typing import Tuple, Literal, TypeAlias
from dataclasses import dataclass


Interval = Tuple[int, int]
from ecg_denoising_pipeline.utils import convert_seconds_to_hms, friendly_print_denoising_cfg
from ecg_denoising_pipeline.utils import friendly_print_denoising_cfg_short
from ecg_denoising_pipeline.io_utils import load_gaps, load_array_or_fail
from ecg_denoising_pipeline.config import load_denoising_config_yaml
from ecg_denoising_pipeline.steps import check_denoising_config

# --- ecg_denoising_pipeline imports (as in your original) ---
from ecg_denoising_pipeline import (
    run_denoising_pipeline,
    print_heading,
    as_seconds,
)

from zive_data_utils import (
    find_project_root_by_name,
    intervals_to_hms,
    samples_dict_to_seconds,
    ecg_min_max_stats,
    remap_intervals_after_clipping,
    dotted_unix_to_naive_vilnius,
    read_gaps_json,
    clip_by_day_hour_min_ampm,
    intervals_to_day_hm_ampm,
    index_to_day_hm_ampm,
)

# Start timing
start_time_1 = time.time()

print("\n**********HEART RATE VARIABILITY\n")

HOME = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
PROJECT_ROOT = find_project_root_by_name(target="S-ITP-25-9", start=HOME)

print("PROJECT ROOT_DIR:", PROJECT_ROOT)
print("PROJECT HOME DIR:", HOME)

print("\n*****ECG DENOISING")
print("\nData, Configuration and Model Paths:")
cfg_denoising_path = HOME/'example_data/config_denoising.yaml'
model_unet_dir = PROJECT_ROOT/'MODEL_UNET'
data_dir = PROJECT_ROOT/'DATA/LONG_ECG_AND_SCRIPTS'
gaps_dir = PROJECT_ROOT/'DATA/LONG_ECG_AND_SCRIPTS'

print("DATA_DIR:", data_dir)
print("CONFIG  :", cfg_denoising_path)
print("MODEL_DIR:", model_unet_dir)

        #   HRV CONFIG **************************************

# Load and check hrv config
win_sec=300
step_sec=300
win_sec=60
step_sec=60
noise_thr=0.01
alpha=0.30
print_heading("HRV config")
print(f"win_sec: {win_sec}, step_sec: {step_sec}, noise_thr: {noise_thr}, alpha: {alpha}")

        #   DENOISING PIPELINE CONFIG *************************

print_heading("\nDenoising pipeline config")
cfg_denoising = load_denoising_config_yaml(str(cfg_denoising_path))

#   LONG PRINT of denoising config 
# friendly_print_denoising_cfg(cfg_denoising)
#   SHORT PRINT of denoising config
friendly_print_denoising_cfg_short(cfg_denoising)

check_denoising_config(cfg_denoising)
fs = cfg_denoising.fs

        #   INPUT DATA **************************************
 # start0
# Input data
file_name = "Jancoriene_23_51_no_gaps.npy"  # xxx sek. labai daug ekstrasistolių
file_name = "daunoraviciene_12_57_no_gaps.npy"  # xxx sek.
file_name = "klaipedos_triuksmai_18_36_no_gaps.npy" # labai daug triukšmų 
# FileName = "klaipedos_triuksmai_18_36.111"  # 66965 sek.
# pause_indices = [[332800,333000],[588999,600200],[676999,677000],[932999,942600],[980999,991200],[1067999,1078000],[1116399,1121000],[1402599,1427400],[1491399,1493600],[1800799,1806200],[1844599,1871200],[3215195,3279600],[3369199,3409600],[3447999,3513400],[3551799,3594200],[3978200,4016800],[4144799,4193000],[4256999,4262800],[4608400,4622000],[4660399,4731600],[4782799,4797200],[7562002,7608800],[7800799,7813400],[7851799,7926000],[9845998,9931800],[9982999,10137400],[10226999,10237200],[10288399,10345400],[10447799,10536000],[10663999,10678600],[10716999,10845000],[12624198,12658400],[13003999,13011600],[13165199,13173600],[13250399,13354600]]
# zive@kulig.lt

file_name = "ignas_3_15.npy"
file_name = "juskevicius_3_14_no_gaps.npy"
file_name = "Jancoriene_23_51.npy"
file_name = "1065_11.npy"

print_heading("Input data")
print(f"File: {file_name}")

# Load ECG signal and json file with first record name and gaps info
x = load_array_or_fail(data_dir, file_name)
# x = x[:9*60*60*200]  # use only first 9 hours for faster testing

len_secs = math.ceil(len(x) / cfg_denoising.fs)
h, m, s = convert_seconds_to_hms(len_secs)
print(f"\nLoaded full ECG signal: len(ecg): {len(x)} samples (~{len_secs:.1f} s) duration: {h:02d}:{m:02d}:{s:02d}")

path = (Path(data_dir) / file_name).with_suffix(".gaps")
first_record_name, gaps_indices = read_gaps_json(path)

        #   START DATETIME AND GAPS **************************************

print("first_record_name from gaps file:", first_record_name)
start_dt = dotted_unix_to_naive_vilnius(first_record_name)
# start_dt = datetime(2025, 12, 1, 0, 0, 0) 
end_dt = start_dt + timedelta(seconds=len_secs)
print("start_dt:", start_dt.strftime("%Y-%m-%d %I:%M %p").lstrip("0"))
print("end_dt  :", end_dt.strftime("%Y-%m-%d %I:%M %p").lstrip("0"))

# Basic stats
def print_ecg_basic_stats(x, fs, start_dt):
    xmin, xmax, imin, imax = ecg_min_max_stats(x)
    print(f"min: {xmin:.2f} at DD:HH:MM: {index_to_day_hm_ampm(imin, fs=fs, start_dt=start_dt)}")
    print(f"max: {xmax:.2f} at DD:HH:MM: {index_to_day_hm_ampm(imax, fs=fs, start_dt=start_dt)}")

print("\nECG full original (not filtered) basic stats:")
print_ecg_basic_stats(x, fs, start_dt)

print("\nLoaded gaps from path:", path)
print(f"Loaded {len(gaps_indices)} gap intervals.")
print("Gaps (in samples):", gaps_indices)
print("Gaps (as DD:HH:MM):", intervals_to_day_hm_ampm(gaps_indices, cfg_denoising.fs, start_dt))

        #   CLIPPING **************************************
i0 = 0
i1 = 0
flag_clipping = False
if flag_clipping:
    # Example usage:
    # clip_dhm_start=(27, 8, 30, "AM")
    # clip_dhm_end=(27, 12, 30, "PM")

    ClipDHM = (27, 8, 30, "AM")  # (day, hour, minute, "AM"/"PM")
    ClipDHM_end = (27, 12, 30, "PM")
    
    ClipDHM = (1, 12, 5, "AM")  # (day, hour, minute, "AM"/"PM")
    ClipDHM_end = (1, 12, 6, "AM")
    
    clip_x, clip_dt_start, clip_dt_end, i0, i1 = clip_by_day_hour_min_ampm(
        x,
        cfg_denoising.fs,
        start_dt,
        end_dt,
        ClipDHM,
        ClipDHM_end
    )
     
    print(f"\nECG signal clipped")
    len_secs = math.ceil(len(clip_x) / cfg_denoising.fs)
    h, m, s = convert_seconds_to_hms(len_secs)
    print(f"Clipped original ECG signal: len(ecg): {len(clip_x)} samples (~{len_secs:.1f} s) duration: {h:02d}:{m:02d}:{s:02d}")
    print("clip_start_dt:", clip_dt_start.strftime("%Y-%m-%d %I:%M %p").lstrip("0"))
    print("clip_end_dt:", clip_dt_end.strftime("%Y-%m-%d %I:%M %p").lstrip("0"))
    start_dt = clip_dt_start  # update start_dt for further processing
    x = clip_x  # use clipped signal 
    # print(f"ECG signal to time of day: {clip_start_dt.time()} - {clip_end_dt.time()}")
    
    # Basic stats
    print("\nECG clipped basic stats:")
    print_ecg_basic_stats(x, fs, start_dt)

            #   CLIPPING GAPS *************************************


if flag_clipping:
    # Map gaps to clipped signal indices
    gaps_indices_clipped = remap_intervals_after_clipping(gaps_indices, i0, i1)
    gaps_indices = gaps_indices_clipped
    print(f"\nMapped gaps to clipped signal indices.")
    print("Gaps clipped (in samples):", gaps_indices)
    print("Gaps (as DD:HH:MM):", intervals_to_day_hm_ampm(gaps_indices, cfg_denoising.fs, start_dt))

        # DENOISING **************************************

print("\nRunning Denoising pipeline...")
# Execute the pipeline in the notebook with explicit arguments
res_denoising, cfg_denoising = run_denoising_pipeline(
    x=x,
    gaps_indices=gaps_indices,
    config_path=cfg_denoising_path,
    model_dir=model_unet_dir,
    disable_motions=False,
)

        # RESULTS **************************************

print_heading("ECG DENOISING pipeline results")
print(f"len_original: {len(res_denoising.ecg_orig)}")
print(f"len_start   : {len(res_denoising.ecg_start)}") 
print(f"len_denoised   : {len(res_denoising.ecg_denoised)}")

print("\nMaps (sample intervals):")
print("map_gaps    :", res_denoising.map_gaps)
print("map_outliers:", res_denoising.map_outliers)
print("map_rdropouts:", res_denoising.map_rdropouts)
print("map_motions :", res_denoising.map_motions)

print_heading("Detected intervals (in samples)")
print("Outliers (start):", res_denoising.outliers_indices_start)
print("Rdropouts (nout):", res_denoising.rdropouts_indices_nout)
print("Motions (nrd)   :", res_denoising.motions_indices_nrd)

print_heading("Detected intervals (in seconds)")
print("Outliers (start):", as_seconds(res_denoising.outliers_indices_start, cfg_denoising.fs))
print("Rdropouts (nout):", as_seconds(res_denoising.rdropouts_indices_nout, cfg_denoising.fs))
print("Motions (nrd)   :", as_seconds(res_denoising.motions_indices_nrd, cfg_denoising.fs))

print_heading("Detected intervals (as HH:MM:SS, using fixed start_dt)")
print("Outliers (start):", intervals_to_hms(res_denoising.outliers_indices_start, cfg_denoising.fs, start_dt))
print("Rdropouts (nout):", intervals_to_hms(res_denoising.rdropouts_indices_nout, cfg_denoising.fs, start_dt))
print("Motions (nrd)   :", intervals_to_hms(res_denoising.motions_indices_nrd,   cfg_denoising.fs, start_dt))

print("\nProjected intervals (in samples):")
print("projected_to_orig", res_denoising.projected_to_orig)
print("projected_to_start", res_denoising.projected_to_start)

print("\nProjected intervals (in seconds):")
proj_orig_sec  = samples_dict_to_seconds(res_denoising.projected_to_orig,  cfg_denoising.fs,ndigits=3)
proj_start_sec = samples_dict_to_seconds(res_denoising.projected_to_start, cfg_denoising.fs,ndigits=3)

print("projected_to_orig (s):", proj_orig_sec)
print("projected_to_start (s):", proj_start_sec)

end_time_1 = time.time()
time_taken_1 = end_time_1 - start_time_1
print(f"\nTime taken for the DENOISING pipeline: {time_taken_1:.2f} secs")  



In [ ]:

# 2-a dalis: ECTOPY: TEST premature beats detection, naudojant išvalytą nuo triukšmų signalą
# Naudoja configuracijos failą config_ectopy.yaml

from ecg_ectopy_pipeline import run_ectopy_pipeline

# start1
"""
Resolve key folders:
    - HOME: the project subfolder that contains CONFIG and sibling DATA/MODEL_VU_CNN
    - ROOT: project root (parent of HOME)
    """
HOME = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
ROOT = HOME.parents[0]    # S-ITP-25-9
print("ROOT:", ROOT)

# Configuration and model directories
cfg_path = ROOT / "CONFIG" / "ectopy_config.yaml"
model_dir = ROOT / "MODEL_VU_CNN"  # S-ITP-25-9/MODEL_UNET

print("CFG_PATH :", cfg_path)
print("MODEL_DIR:", model_dir)


# "Run ECG ectopy pipeline on a signal from a ecg_denoising_pipeline
# Execute the pipeline in the notebook with explicit arguments
res_ectopy = run_ectopy_pipeline(
    file_name=file_name,
    fs=int(cfg_denoising.fs),
    res_denoising = res_denoising,
    ectopy_config_path=cfg_path,
    ectopy_model_dir=model_dir,
    disable_ectopy_removing=False,
    show_debug_plots=False,
    portion_to_plot_secs=10.,
)

# Results
print_heading("ECTOPY pipeline results:")
print("rpeaks_on_denoised_df head (first 100 rows):")
print(res_ectopy.rpeaks_on_denoised_df.head(100))


In [ ]:
# Vaizdavimas su annotacijomis ir predikcijomis

# Vaizduojame signalą su ectopijomis : res.ecg_start
# naudojame rpeaks_on_start_df (pred_df) ir annot_df
# Show denoised signal with ectopy //////////////////////////////////

from __future__ import annotations

from dataclasses import dataclass
from typing import Optional, List, Sequence
import numpy as np
import pandas as pd

import numpy.typing as npt

from ecg_denoising_pipeline import map_mark_denoised_to_start, load_nonzero_annots
from ecg_denoising_pipeline.plotting import (
    plot_ecg_annot_pred,
    divide_signal_into_fragments,
    plot_gap_legend,
)

IntArray1D = npt.NDArray[np.int_]
BoolArray1D = npt.NDArray[np.bool_]

fs = cfg_denoising.fs

flag_show_denoised_with_ectopy = True
# Show denoised signal with ectopy //////////////////////////////////

if flag_show_denoised_with_ectopy:
    
    print("\nShow final signal with ectopy annotations and predictions:")
    
    # Prepare data for plotting

    # 1. Iš išvalyto nuo triukšmų signalo randame R-peak'us rpeaks_on_denoised
    #  ir remapiname į R-peak'us į pradinį signalą
    # Ensure we pass a List[int] (indices) to map_mark_denoised_to_start.
    rpeak_col = res_ectopy.rpeaks_on_denoised_df["rpeak"]
    if rpeak_col.dtype == bool:
        # Convert boolean mask to indices
        rpeaks_on_denoised: Sequence[int] = np.flatnonzero(rpeak_col.values).tolist()
    else:
        # Already indices; enforce int list
        rpeaks_on_denoised: Sequence[int] = rpeak_col.astype(int).tolist()
    
    rpeaks_on_start, rpeaks_on_gap = map_mark_denoised_to_start(rpeaks_on_denoised, res_denoising)
    rpeaks_on_start = np.array(rpeaks_on_start, dtype=int)

    # 2. Jei yra ectopy annotacijos,jas užkrauname
    annot_df_: Optional[pd.DataFrame] = load_nonzero_annots(data_dir, file_name)
    if annot_df_ is None:
        print("\nNo non-zero annotations found.")
    else:
        print(f"\nlen(annot_df_): {len(annot_df_)}")

    # 3. Remapiname rpeaks_on_denoised_df, gautą iš get_beats_ml_classes į start signalą
    # Build dataframe in 'start' coordinates (keeps Index and pred)
    rpeaks_on_start_df = res_ectopy.rpeaks_on_denoised_df.copy()
    rpeaks_on_start_df["rpeak"] = rpeaks_on_start

    print(f"Mapped {len(rpeaks_on_start_df)} R-peaks from final -> start")
    pred_df_ = rpeaks_on_start_df.loc[rpeaks_on_start_df["pred"].ne(0)].copy()
    if pred_df_.empty:
        pred_df_ = None

    START_SECS = 0.0
    END_SECS = len(x) / fs
    PORTION_LENGTH_SECS = 20.
    DISPLAY_AS_SECONDS = True
    SAVE_PLOTS = False
    PLOT_DIR = HOME / "PLOTS" / "ectopy_plots"
    PLOT_SUFFIX = "with_ectopy"
    fileName = "ecg_start"

    ecg_signal = res_denoising.ecg_start

    rpeaks_on_start_secs = np.array([x/fs for x in rpeaks_on_start])
    print(f"rpeaks_start (len {len(rpeaks_on_start)}): {len(rpeaks_on_start_secs)}")

    # annot_df_ = annot_df.loc[annot_df["annot"] != 0].copy()
    # pred_df = rpeaks_on_start_df
    # pred_df_ = pred_df.loc[pred_df["pred"] != 0].copy()

    # Convert outliers indices to seconds for reporting
    outliers_idx_seconds = [(s / fs, e / cfg_denoising.fs) for (s, e) in res_denoising.projected_to_start['outliers']]
    print(f"outliers_idx_seconds: {outliers_idx_seconds}")
    rdropouts_idx_seconds = [(s / fs, e / cfg_denoising.fs) for (s, e) in res_denoising.projected_to_start['rdropouts']]
    print(f"rdropouts_idx_seconds: {rdropouts_idx_seconds}")
    motions_idx_seconds = [(s / fs, e / cfg_denoising.fs) for (s, e) in res_denoising.projected_to_start['motions']]
    print(f"motions_idx_seconds: {motions_idx_seconds}")

    fragment_samples = divide_signal_into_fragments(ecg_signal, int(PORTION_LENGTH_SECS * fs))
    fragment_secs = [(start / fs, end / fs) for start, end in fragment_samples]
    print("Fragments (s):", [(round(s, 1), round(e, 1)) for s, e in fragment_secs])
 
    # Plot small rectangles and notes in one line
    plot_gap_legend('outliers', 'rdropouts', 'motions')
    print("\noutliers-> yellow, rdropouts-> blue, motions-> red")

    # gap_layers = [
    #     GapLayer(tuple(gap1_indices_secs or []), "red", 0.5),
    #     GapLayer(tuple(gap2_indices_secs or []), "blue", 0.5),
    #     GapLayer(tuple(gap3_indices_secs or []), "yellow", 0.7),
    # ]

 
    # next_fragment_idx = 1
    # for start_sec, end_sec in fragment_secs:
    plot_ecg_annot_pred(
        fileName=fileName,
        ecg_signal=ecg_signal,
        fs=fs,
        plot_signal_from_in_secs=START_SECS,
        plot_signal_to_in_secs=END_SECS,
        portion_length_in_secs=PORTION_LENGTH_SECS,
        plot_save_dir=Path(PLOT_DIR) if SAVE_PLOTS else None,
        save_mark=PLOT_SUFFIX,
        recID=None,
        gap1_indices_secs=motions_idx_seconds, # red
        gap2_indices_secs=rdropouts_idx_seconds, # blue
        gap3_indices_secs=outliers_idx_seconds, # yellow
        rpeak_indices_secs=rpeaks_on_start_secs,
        annot_df=annot_df_,
        pred_df=pred_df_,
        tol_samples=10,
        flag_secs=DISPLAY_AS_SECONDS,
    )

# def plot_ecg_annot_pred(
#     fileName: str | None,
#     ecg_signal: np.ndarray,
#     fs: int,
#     plot_signal_from_in_secs: float,
#     plot_signal_to_in_secs: float,
#     portion_length_in_secs: float,
#     plot_save_dir: Path | None = None,
#     save_mark: Optional[str] = None,
#     recID: Optional[int] = None,
#     gap1_indices_secs: Optional[Sequence[SecondsInterval]] = None,
#     gap2_indices_secs: Optional[Sequence[SecondsInterval]] = None,
#     gap3_indices_secs: Optional[Sequence[SecondsInterval]] = None,
#     annot_df: Optional[pd.DataFrame] = None,
#     pred_df: Optional[pd.DataFrame] = None,
#     tol_samples: int = 10,
#     rpeak_indices_secs: Optional[np.ndarray] = None,
#     flag_secs: bool = True,
# ):
    print("Finished plotting", file_name)

In [ ]:
# Įvertiname ectopijų aptikimo tikslumą naudojant annotacijas ir predikcijas


# Klaidos skaičiavimas
# dalis paimta iš TEST_VU_CNN/zive_aritmijos_klasifikacija_NN_algoritmas_one_keras3.ipynb

from ecg_denoising_pipeline import read_df_annot
from ecg_ectopy_pipeline import (
    merge_rpeaks_with_annotations,
    print_classification_results,
    evaluate_binary_classification,
)

# from step_beats_no_ectopy import map_rpeaks_denoised_to_start


# +++++++  RPIKŲ IR EKSTRASYSTOLIŲ INDEKSŲ PERSKAIČIAVIMAS Į PRADINĮ SIGNALĄ +++++++++++++++++++

# rpikų indeksų su klasių numeriais perskaičiavimas į pradinių duomenų signalą ecg_start

# rezultatas: rpeaks_on_start_df

# pirmiausiai perskaičiuojame rpeaks_on_denoised į rpeaks_on_start
# naudojame žemėlapius iš denoising pipeline: res.map_motions, res.map_rdropouts, res.map_outliers, res.map_gaps

# nuskaitome anotoutacijas iš medikų  ir sulyginame su ML klasifikacija

# 1. Iš išvalyto nuo triukšmų signalo randame R-peak'us rpeaks_on_denoised
#  ir remapiname į R-peak'us į pradinį signalą
rpeaks_on_denoised = res_ectopy.rpeaks_on_denoised_df["rpeak"].astype(int).tolist()
rpeaks_on_start, rpeaks_on_gap = map_mark_denoised_to_start( rpeaks_on_denoised, res_denoising )
rpeaks_on_start = np.array(rpeaks_on_start, dtype=int)

# Build dataframe in 'start' coordinates (keeps Index and pred)
rpeaks_on_start_df = res_ectopy.rpeaks_on_denoised_df.copy()
rpeaks_on_start_df["rpeak"] = rpeaks_on_start
# Backward-compatible alias if used later in the notebook
rpeaks_df_start = rpeaks_on_start_df

print(f"Mapped {len(rpeaks_on_start_df)} R-peaks from final -> start")
# print("\nrpeaks_on_start_df:")
# print(rpeaks_on_start_df.head(10))

            # ++++++++++++++++++++++++++  ECG PŪPSNIŲ KLASIFIKACIJOS SULYGINIMAS SU MEDIKŲ ANOTACIJOMIS 
 
# Nuskaitome paciento įrašo medikų anotacijas atr_symbol_orig ('N', 'S', 'V', 'U')
# ir jų indeksus atr_sample_orig (rpeaks vietas signal masyve)
print(f"\nfileName: {file_name}")
annot_df = read_df_annot(data_dir, file_name)
print(f"\nlen(annot_df): {len(annot_df)}")
# print("\nannot_df:")
# print(annot_df.head(20))

tolerance = 20
print("\ntolerance:", tolerance)

    # Merge the dataframes
df_matched, df_unmatched, not_found_count, description = merge_rpeaks_with_annotations(rpeaks_on_start_df,
                                                                    annot_df, tolerance=tolerance)

print(description)
print(f"\nlen(df_matched): {len(df_matched)}")
df_matched = df_matched.sort_values(by='diff', ascending=False)
# print(df_matched.head(10))
print(f"\nlen(df_unmatched): {len(df_unmatched)}")
df_unmatched["rpeak_sec"] = df_unmatched["rpeak"] / fs
df_unmatched["closest_rpeak_annot_sec"] = df_unmatched["closest_rpeak_annot"] / fs
# print(df_unmatched.head(20))

# Remove annot == 3 from df_matched
removed_count = (df_matched['annot'] == 3).sum()
df_matched = df_matched[df_matched['annot'] != 3].reset_index(drop=True)
print(f"\nRemoved {removed_count} rows with annot == 3")

# (unique_labels_annot, counts_annot) = np.unique(df_matched['annot'].values, return_counts=True)
counts = df_matched['annot'].value_counts(dropna=False)
unique_labels_annot = counts.index.to_numpy()
counts_annot = counts.to_numpy()
print("\nLabels for annot: ", unique_labels_annot, counts_annot, "Total:", counts_annot.sum())

# (unique_labels_pred, counts_pred) = np.unique(df_matched['pred'].values, return_counts=True)
counts = df_matched['pred'].value_counts(dropna=False)
unique_labels_pred = counts.index.to_numpy()
counts_pred = counts.to_numpy()
print("Labels for pred: ", unique_labels_pred, counts_pred, "Total:", counts_pred.sum())


# Surandame klasifikavimo tikslumą ir išvedame rezultatus
print("\nKlasifikavimo tikslumas")
comment = []
test_labels = df_matched['annot'].values
pred_labels = df_matched['pred'].values
print_classification_results(test_labels, pred_labels, comment)

# Sulyginimui su binarine klasifikacija perskaičiuojame labels: N=0, S=1, V=1, U=1
# Ensure ndarray to avoid cases wHOME a scalar bool leaks in and lacks `.astype`
test_labels_bin = (np.asarray(test_labels) != 0).astype(int)
pred_labels_bin = (np.asarray(pred_labels) != 0).astype(int)
print("\nClassification results for binary case")
evaluate_binary_classification(test_labels_bin, pred_labels_bin, positive_class=1)



In [ ]:
# Vaizduojame švarų signalą be ectopijų //////////////////////////////////

from ecg_denoising_pipeline import get_rpeaks

flag_show_denoised_without_ectopy = True

# Švarus signalas be ectopijų: res.ecg_denoised_no_ectopy

if flag_show_denoised_without_ectopy:
    
    print("\nShow denoised signal without ectopy annotations and predictions:")
    
    # Prepare data for plotting

    # 1. Iš išvalyto nuo triukšmų ir ekstrasistoliųs signalo randame R-peak'us rpeaks_on_no_ectopy
    ecg_signal = res_ectopy.ecg_no_ectopy
    # rpeaks_on_no_ectopy = map_rpeaks_denoised_to_start(
    rpeaks_on_no_ectopy, rpeaks_on_no_ectopy_secs = get_rpeaks(ecg_signal, fs=fs)

    PORTION_LENGTH_SECS = 20.
    START_SECS = 0.0
    END_SECS = len(ecg_signal) / fs
    DISPLAY_AS_SECONDS = True
    SAVE_PLOTS = False
    PLOT_DIR = HOME / "PLOTS" / "ectopy_plots"
    PLOT_SUFFIX = "without_ectopy"
    fileName = "ecg_denoised_no_ectopy"

    print(f"rpeaks_start (len {len(rpeaks_on_no_ectopy)}): {len(rpeaks_on_no_ectopy_secs)}")

    fragment_samples = divide_signal_into_fragments(ecg_signal, int(PORTION_LENGTH_SECS * fs))
    fragment_secs = [(start / fs, end / fs) for start, end in fragment_samples]
    print("Fragments (s):", [(round(s, 1), round(e, 1)) for s, e in fragment_secs])
 
    # next_fragment_idx = 1
    # for start_sec, end_sec in fragment_secs:
    plot_ecg_annot_pred(
        fileName=fileName,
        ecg_signal=ecg_signal,
        fs=fs,
        plot_signal_from_in_secs=START_SECS,
        plot_signal_to_in_secs=END_SECS,
        portion_length_in_secs=PORTION_LENGTH_SECS,
        plot_save_dir=Path(PLOT_DIR) if SAVE_PLOTS else None,
        save_mark=PLOT_SUFFIX,
        recID=None,
        rpeak_indices_secs=rpeaks_on_no_ectopy_secs,
        tol_samples=0,
        flag_secs=DISPLAY_AS_SECONDS,
    )

    print("Finished plotting", fileName)
